# Agent Observability Experiments Walkthrough

This notebook runs a small deterministic Agent Observability experiment. It shows the suite shape, optional stored-suite push, agent invocation, generation recording, score export, finalization, and report fetch.

This feature runs against Grafana Cloud. Before launching the notebook, configure the Grafana Cloud ingest settings shown in the example README. To push or pull stored suites, also configure the Grafana stack control endpoint and service-account token.

In [ ]:
import json
import os
import time
from dataclasses import asdict
from pathlib import Path

from agento11y import experiments

def required_env(name: str) -> str:
    value = os.environ.get(name, "").strip()
    if not value:
        raise RuntimeError(f"Set {name} to the value from your Grafana Cloud stack before running this notebook")
    return value


endpoint = required_env("AGENTO11Y_ENDPOINT")
tenant_id = required_env("AGENTO11Y_AUTH_TENANT_ID")
ingest_token = required_env("AGENTO11Y_AUTH_TOKEN")
grafana_url = required_env("AGENTO11Y_GRAFANA_URL")

run_id = os.environ.get("AGENTO11Y_EXPERIMENT_ID", f"notebook-suite-{int(time.time())}")
print("experiment_id=", run_id)
print("endpoint=", endpoint)


## 1. Define Or Load A Suite

Portable suites use `suite_id`, optional `version`, and `cases`. Each case uses the short `id` field locally; the SDK translates it to the backend `test_case_id` field when pushing to stored suites or creating trials.

In [ ]:
suite_path = Path("evals/dashboard-regression.yaml")
suite = experiments.TestSuite.from_yaml(str(suite_path))
print(suite.to_yaml())
print("cases=", [case.id for case in suite.cases])


## 2. Optionally Push The Suite

This writes suite metadata to the Grafana Cloud control plane, creates or reuses a draft version, and upserts its cases. If control-plane credentials are not configured, the experiment can still run with the suite loaded from YAML.

In [ ]:
try:
    suites = experiments.TestSuitesClient()
    pushed = suites.push_suite(
        suite,
        publish=os.environ.get("AGENTO11Y_PUBLISH_SUITE", "false").lower() == "true",
        changelog="pushed from experiment walkthrough notebook",
    )
    suite = pushed.suite
    print(f"stored suite: {pushed.suite_id}@{pushed.suite_version}, published={pushed.published}")
except Exception as exc:
    print("stored-suite push skipped:", type(exc).__name__, exc)
    print(f"using local suite: {suite.suite_id}@{suite.version}")


## 3. Run The Agent And Write Experiment Data

For each test case, the trial context creates a typed trial. The notebook then calls a tiny deterministic agent, records a candidate generation, binds that generation to the trial, emits a final score, and writes a small JSON artifact. Leaving the experiment context finalizes the run.

In [ ]:
def run_agent(case: experiments.TestCase) -> str:
    data = case.input if isinstance(case.input, dict) else {"request": case.input}
    request = str(data.get("request", ""))
    return "Generated dashboard summary: " + request


def expected_terms(case: experiments.TestCase) -> list[str]:
    expected = case.expected
    if isinstance(expected, dict) and isinstance(expected.get("must_include"), list):
        return [str(term).lower() for term in expected["must_include"]]
    return []


verifier = experiments.Evaluator(
    evaluator_id="notebook.expected_terms",
    version="1",
    kind="deterministic",
)

with experiments.experiment(
    name="Notebook suite walkthrough",
    experiment_id=run_id,
    suite=suite,
    candidate={"agent_name": "notebook-agent", "agent_version": "example", "model_name": "deterministic"},
    tags=["notebook", "suite-walkthrough"],
) as exp:
    for case in suite.cases:
        with exp.trial(case) as trial:
            output = run_agent(case)
            terms = expected_terms(case)
            matched = [term for term in terms if term in output.lower()]
            score = len(matched) / len(terms) if terms else 1.0
            passed = score >= 1.0

            conversation_id = experiments.stable_id("conv", exp.experiment_id, case.id, trial.ref.attempt)
            exp.client.record_generation(
                trial.generation_id,
                conversation_id=conversation_id,
                input_text=json.dumps(case.input),
                output_text=output,
                model_provider="deterministic",
                model_name="deterministic",
                agent_name="notebook-agent",
                operation_name="run_agent",
                tags={"experiment.run_id": exp.experiment_id, "task_id": case.id},
                metadata={"suite_id": suite.suite_id, "suite_version": suite.version},
            )
            trial.bind_generation(trial.generation_id, conversation_id=conversation_id)
            trial.score(
                "final",
                score,
                passed=passed,
                explanation=f"matched {len(matched)}/{len(terms)} expected terms",
                evaluator=verifier,
                generation_id=trial.generation_id,
                metadata={"matched_terms": matched, "expected_terms": terms},
            )
            trial.artifact(
                "notebook-result",
                data={"case_id": case.id, "output": output, "score": score, "passed": passed},
                kind="json",
            )

    report = exp.report()

print("accepted_scores=", exp.accepted_scores)
print("url=", exp.url)
print("suite=", report.run.suite_id, report.run.suite_version)
print("pass_rate=", report.summary.pass_rate, "mean_score=", report.summary.final_score_avg)


## 4. Inspect What Was Written

The report comes from `GET /api/v1/eval/experiments/{experiment_id}/report`. Rows are grouped by test case, and each trial contains the score/artifact/generation links created above.

In [ ]:
client = experiments.Client(
    endpoint,
    tenant_id=tenant_id,
    ingest_token=ingest_token,
    grafana_url=grafana_url,
)
latest_report = client.get_report(run_id)
scores, _ = client.list_scores(run_id)
print(json.dumps({
    "experiment_id": latest_report.run.run_id,
    "status": latest_report.run.status,
    "suite_id": latest_report.run.suite_id,
    "suite_version": latest_report.run.suite_version,
    "candidate": latest_report.run.candidate,
    "summary": asdict(latest_report.summary),
    "rows": [row.get("test_case_id") for row in latest_report.rows],
    "score_count": len(scores),
}, indent=2, default=str))
